# 🎓 Proyecto de práctica — Databricks Certified Data Engineer Associate

**Objetivo:** construir un pipeline end-to-end (bronze → silver → gold) que cubra los 7 dominios del examen, usando un dataset sintético de e-commerce.

---

## 📐 Arquitectura

CRM / Sistema de ventas (fuente externa)
│
▼
01_landing (archivos crudos: CSV + JSON)
│
▼ COPY INTO (batch) / Auto Loader (streaming)
02_bronze (copia fiel de la fuente, todo como STRING)
│
▼ limpieza, casteo de tipos, explode(), AUTO CDC
03_silver (datos limpios, tipados, con historial)
│
▼ agregaciones, materialized views
04_gold (listo para BI / analítica)


## 📦 Catálogo

| Schema | Contenido |
|---|---|
| `01_landing` | Archivos crudos (Volumes: `dim_tables`, `orders_landing`) |
| `02_bronze` | Copia fiel de la fuente, sin transformar |
| `03_silver` | Datos limpios, tipados, `order_items` explotado, CDC de customers |
| `04_gold` | Materialized views agregadas para BI |
| `05_ops` | Checkpoints de streaming (no es capa de datos) |

## 🔑 Convenciones importantes

- El catálogo (`data-engineer-associate`) y los esquemas numerados (`02_bronze`, etc.) tienen guiones/números → **siempre usar backticks** en SQL: `` `{CATALOG}`.`02_bronze` ``
- Bronze = todo `STRING`, sin castear tipos ni validar — fiel a la fuente
- `customers_v1` / `customers_v2` = dos snapshots del mismo CRM en momentos distintos, usados juntos para `AUTO CDC` (SCD Type 2)
- `items` llega como JSON anidado dentro de cada orden → se aplana en silver con `from_json()` + `explode()` para crear `order_items`

---

## ✅ Progreso

- [x] Datos sintéticos generados (Faker, semillas fijas) y subidos a `01_landing`
- [x] Bronze: `stores_bronze` (20 filas), `products_bronze` (200 filas)
- [x] Bronze: `orders_bronze` (500,000 filas) vía Auto Loader
- [ ] Silver: `stores`, `products` (casteo de tipos)
- [ ] Silver: `order_items` (`from_json` + `explode`)
- [ ] Silver: `orders` (encabezado limpio)
- [ ] Silver: `customers` (`AUTO CDC` / SCD Type 2)
- [ ] Gold: materialized views (ventas por región/producto/mes)
- [ ] Lakeflow Job (orquestación end-to-end)
- [ ] CI/CD con Databricks Asset Bundles
- [ ] Troubleshooting: forzar y diagnosticar skew en Spark UI
- [ ] Gobernanza: GRANT/REVOKE, column masking, row-level security

---
*Preparación para el examen Databricks Certified Data Engineer Associate (versión mayo 2026)*

In [0]:
CATALOG = "data-engineer-associate"

# ------------------------------------------------------------------
# STORES
# ------------------------------------------------------------------

# TODAS las columnas como STRING. Bronze debe ser una copia fiel
# del dato crudo -- no casteamos tipos aquí. El casteo real (a INT,
# DOUBLE, etc.) lo hacemos después en la capa silver, a propósito.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`02_bronze`.stores_bronze (
    store_id STRING,
    region STRING,
    country STRING
)
""")

spark.sql(f"""
COPY INTO `{CATALOG}`.`02_bronze`.stores_bronze
FROM '/Volumes/{CATALOG}/01_landing/dim_tables/stores.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
""")
# Sin 'inferSchema' = 'true': ya no necesitamos que Spark adivine tipos,
# porque el destino es STRING y el CSV se lee como texto de todas formas.
# Con la tabla ya declarada en STRING, no hay choque de tipos posible.

# ------------------------------------------------------------------
# PRODUCTS
# ------------------------------------------------------------------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`02_bronze`.products_bronze (
    product_id STRING,
    name STRING,
    category STRING,
    price STRING
)
""")

spark.sql(f"""
COPY INTO `{CATALOG}`.`02_bronze`.products_bronze
FROM '/Volumes/{CATALOG}/01_landing/dim_tables/products.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
""")

print("stores_bronze:", spark.table(f"`{CATALOG}`.`02_bronze`.stores_bronze").count(), "filas")
print("products_bronze:", spark.table(f"`{CATALOG}`.`02_bronze`.products_bronze").count(), "filas")

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`05_ops`.checkpoints")

checkpoint_path = f"/Volumes/{CATALOG}/05_ops/checkpoints/orders_autoloader_checkpoint"
schema_path = f"/Volumes/{CATALOG}/05_ops/checkpoints/orders_schema"


# --- Definición del STREAM DE LECTURA ---
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "false")   # bronze en crudo: sin castear tipos todavía
    .load(f"/Volumes/{CATALOG}/01_landing/orders_landing/")
)

# --- Definición del STREAM DE ESCRITURA ---
(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(f"`{CATALOG}`.`02_bronze`.orders_bronze")
)

display(spark.table(f"`{CATALOG}`.`02_bronze`.orders_bronze").limit(5))
print("orders_bronze:", spark.table(f"`{CATALOG}`.`02_bronze`.orders_bronze").count(), "filas")

In [0]:
print("orders_bronze:", spark.table(f"`{CATALOG}`.`02_bronze`.orders_bronze").count(), "filas")